# L12c Lab: Model-Based Value Iteration versus Model-Free Q-Learning

> **Learning objectives**
>
> - Solve one environment with a transition model and with sampled experience.
> - Compare greedy policies on nonterminal states.
> - Compare evaluated returns instead of demanding identical tables.
> - Identify the information advantage held by value iteration.


## Setup


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


In [2]:
world = learning_gridworld(slip = 0.1, gamma = 0.95)
model_based = value_iteration_reference(world)
model_free = q_learning(world; episodes = 20_000, seed = 5800)
agreement = policy_agreement(model_based.policy, model_free.policy, world.terminal)

value_evaluation = evaluate_policy(world, model_based.policy; episodes = 5000, seed = 5800)
q_evaluation = evaluate_policy(world, model_free.policy; episodes = 5000, seed = 5800)

comparison = DataFrame(method = ["value iteration", "Q-learning"],
    transition_model_required = [true, false],
    mean_start_return = [value_evaluation.mean_return, q_evaluation.mean_return])
pretty_table(comparison)
(policy_agreement = agreement, value_iterations = model_based.iterations)


┌─────────────────┬───────────────────────────┬───────────────────┐
│          method │ transition_model_required │ mean_start_return │
│          String │                      Bool │           Float64 │
├─────────────────┼───────────────────────────┼───────────────────┤
│ value iteration │                      true │          0.533956 │
│      Q-learning │                     false │          0.533956 │
└─────────────────┴───────────────────────────┴───────────────────┘


(policy_agreement = 0.8888888888888888, value_iterations = 42)

In [3]:
@test model_based.converged
@test agreement >= 0.85
@test abs(value_evaluation.mean_return - q_evaluation.mean_return) < 0.05
:model_based_model_free_comparison_verified


:model_based_model_free_comparison_verified

## Interpretation

Value iteration gets exact access to every transition probability during each backup. Q-learning gets one sampled next state at a time. Similar policies are therefore evidence that sufficient experience recovered the important action ordering—not that the algorithms used the same information.
